# 03 Paper Fig. 4 Style Transport And Mobility

This notebook reproduces the role of paper Fig. 4 with DeePTB-native SERTA mobility APIs: carrier density versus chemical potential, conductivity versus chemical potential, mobility versus carrier density, and mobility versus temperature. It reuses the reduced-grid EPC data produced in notebook 02 when present, and otherwise generates the same reduced-grid data inline.


In [ ]:
from pathlib import Path
import json
import numpy as np

from dptb.nn.dftbsk import DFTBSK
from dptb.postprocess.unified import TBSystem
from dptb.postprocess.unified.eph import Phonons

ROOT = Path.cwd()
DATA = ROOT / "data"
WORK = ROOT / "work"
WORK.mkdir(exist_ok=True)

SK_DATA = DATA / "skf" / "matsci-0-3"
STRUCTURE = ROOT / "graphene.vasp"
PHONOPY_YAML = DATA / "graphene" / "phonons" / "phonopy_disp.yaml"
FORCE_SETS = DATA / "graphene" / "phonons" / "FORCE_SETS"

for path, message in [
    (SK_DATA / "C-C.skf", "The bundled matsci-0-3 C-C.skf file is required."),
    (STRUCTURE, "The bundled graphene.vasp structure is required."),
    (PHONOPY_YAML, "The bundled phonopy_disp.yaml file is required."),
    (FORCE_SETS, "The bundled FORCE_SETS file is required."),
]:
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing. {message}")

BASIS = {"C": ["2s", "2p"]}
model = DFTBSK(
    basis=BASIS,
    skdata=str(SK_DATA),
    overlap=True,
    dtype="float64",
    device="cpu",
    interp_method="smooth_intp",
    r_max={"C": 6.349479778742587},
)
model.eval()
system = TBSystem(data=str(STRUCTURE), calculator=model)


def regularize_tiny_negative_frequencies(phonons, tol=1e-3):
    frequencies = np.array(phonons.frequencies, copy=True)
    min_frequency = float(np.min(frequencies))
    if min_frequency < -tol:
        raise ValueError(
            f"Found phonon frequency {min_frequency} THz below tolerance {-tol} THz; "
            "this looks like a real imaginary mode, not acoustic numerical noise."
        )
    clipped = int(np.count_nonzero(frequencies < 0.0))
    frequencies[frequencies < 0.0] = 0.0
    if clipped:
        phonons = Phonons(
            qpoints=phonons.qpoints,
            frequencies=frequencies,
            eigenvectors=phonons.eigenvectors,
            masses=phonons.masses,
            cell=phonons.cell,
            scaled_positions=phonons.scaled_positions,
            metadata={
                **phonons.metadata,
                "negative_frequency_regularization": "clipped_to_zero",
                "negative_frequency_tolerance_thz": tol,
                "negative_frequency_min_before_clip_thz": min_frequency,
                "negative_frequency_clipped_count": clipped,
            },
        )
    return phonons


def get_eigenvalues_array(kpoints):
    _, evals = system.get_eigenvalues(k_points=np.asarray(kpoints, dtype=float))
    if hasattr(evals, "detach"):
        evals = evals.detach().cpu().numpy()
    return np.asarray(evals, dtype=float)


def json_default(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return str(value)

print(system.atoms)
print(system.model.name, system.model.basis)


In [ ]:
K_POINT = np.array([-2.0 / 3.0, -1.0 / 3.0, 0.0])
G_POINT = np.array([0.0, 0.0, 0.0])
CONDUCTION_BAND = 2
VALENCE_BAND = 1


def q_grid_around_gamma(n=9, qmax=0.0665):
    axis = np.linspace(-qmax, qmax, n)
    qx, qy = np.meshgrid(axis, axis, indexing="xy")
    qpoints = np.column_stack([qx.ravel(), qy.ravel(), np.zeros(qx.size)])
    return qpoints, qx, qy


def choose_k_near_k_for_energy(target_ev=0.30, nscan=41):
    alphas = np.linspace(0.0, 0.18, nscan)
    kpoints = K_POINT[None, :] + alphas[:, None] * (G_POINT - K_POINT)[None, :]
    evals = get_eigenvalues_array(kpoints)
    dirac = float(evals[0, CONDUCTION_BAND])
    energies = evals[:, CONDUCTION_BAND] - dirac
    idx = int(np.argmin(np.abs(energies - target_ev)))
    return kpoints[idx], dirac, energies[idx], evals[idx]


def acoustic_in_plane_modes(phonons, q_index):
    eig = np.asarray(phonons.eigenvectors[q_index])
    freqs = np.asarray(phonons.frequencies[q_index])
    acoustic = np.argsort(freqs)[:3]
    z_weight = np.sum(np.abs(eig[acoustic, :, 2]) ** 2, axis=1)
    in_plane = acoustic[np.argsort(z_weight)[:2]]
    ordered = in_plane[np.argsort(freqs[in_plane])]
    return {"TA": int(ordered[0]), "LA": int(ordered[1])}


def cell_area_ang2(atoms):
    cell = np.asarray(atoms.cell.array, dtype=float)
    return float(np.linalg.norm(np.cross(cell[0], cell[1])))


In [ ]:
from dptb.postprocess.unified.eph import (
    EPCData,
    LinewidthMeshData,
    compute_band_velocities_hamiltonian_derivative,
    compute_linewidth_mesh,
    compute_serta_mobility_scan_si,
)
from dptb.postprocess.unified.eph.data import EPCMeshData

fig3_epc_path = WORK / "fig3_scattering_epc_data.npz"
fig3_linewidth_path = WORK / "fig3_mode_resolved_linewidth.npz"

if not fig3_epc_path.exists() or not fig3_linewidth_path.exists():
    Q_GRID_N = 7
    K_SAMPLE_N = 9
    Q_MAX_FRACTIONAL = 0.0665
    TEMPERATURE_EV = 8.617333262145e-5 * 300.0
    CHEMICAL_POTENTIAL_EV = 0.100
    SIGMA_EV = 0.003
    DISPLACEMENT = 1e-3
    BANDS = [1, 2]

    qpoints, _, _ = q_grid_around_gamma(Q_GRID_N, Q_MAX_FRACTIONAL)
    q_weights = np.full(qpoints.shape[0], 1.0 / qpoints.shape[0])
    alphas = np.linspace(0.01, 0.16, K_SAMPLE_N)
    kpoints = K_POINT[None, :] + alphas[:, None] * (G_POINT - K_POINT)[None, :]
    dirac_for_linewidth = float(get_eigenvalues_array(K_POINT[None, :])[0, CONDUCTION_BAND])
    phonons = regularize_tiny_negative_frequencies(
        Phonons.from_phonopy_file(PHONOPY_YAML, qpoints=qpoints, force_sets_filename=str(FORCE_SETS))
    )
    epc_generated = system.eph.compute_coupling(
        kpoints=kpoints,
        phonons=phonons,
        bands=BANDS,
        displacement=DISPLACEMENT,
        output_npz=fig3_epc_path,
    )
    mesh_generated = EPCMeshData.from_epc_data(
        epc_generated,
        kpoint_weights=np.full(kpoints.shape[0], 1.0 / kpoints.shape[0]),
        qpoint_weights=q_weights,
        metadata={"mesh_kind": "fig4_inline_reduced_intravalley_q_mesh"},
    )
    linewidth_generated = compute_linewidth_mesh(
        mesh_generated,
        chemical_potential=dirac_for_linewidth + CHEMICAL_POTENTIAL_EV,
        temperature=TEMPERATURE_EV,
        sigma=SIGMA_EV,
        broadening="gaussian",
        mode_resolved=True,
    )
    linewidth_generated.save_npz(fig3_linewidth_path)

epc = EPCData.load_npz(fig3_epc_path)
linewidth_modes = LinewidthMeshData.load_npz(fig3_linewidth_path)
linewidth = np.maximum(linewidth_modes.linewidth.sum(axis=-1), 1e-14)

reciprocal_cell = 2.0 * np.pi * np.linalg.inv(np.asarray(system.atoms.cell.array, dtype=float)).T
area = cell_area_ang2(system.atoms)
velocities = compute_band_velocities_hamiltonian_derivative(
    system=system,
    kpoints=epc.kpoints,
    bands=epc.band_indices,
)
dirac_energy = float(get_eigenvalues_array(K_POINT[None, :])[0, CONDUCTION_BAND])

print("kpoints", epc.kpoints.shape)
print("bands", epc.band_indices)
print("area [Angstrom^2]", area)
print("Dirac reference [eV]", dirac_energy)


## Mobility scan matching Fig. 4 panels

The scan uses 2D normalization and SI conversion. Paper-scale transport used a `400 x 400` k mesh and the same `200 x 200` q mesh as Fig. 3; this notebook keeps the reduced mesh from notebook 02, so use the trends and API wiring rather than absolute publication values.


In [ ]:
KB_EV_PER_K = 8.617333262145e-5
MU_REL_EV = np.array([0.03, 0.06, 0.10, 0.16, 0.24, 0.32])
TEMPERATURE_300_EV = KB_EV_PER_K * 300.0
TEMPERATURES_K = np.array([150.0, 200.0, 250.0, 300.0, 400.0])
TEMPERATURES_EV = KB_EV_PER_K * TEMPERATURES_K
TARGET_DENSITY_CM2 = 4.0e12

scan_300 = compute_serta_mobility_scan_si(
    eigenvalues=epc.eigenvalues_k,
    velocities=velocities,
    linewidth=linewidth,
    reciprocal_cell=reciprocal_cell,
    chemical_potentials=dirac_energy + MU_REL_EV,
    temperatures=np.array([TEMPERATURE_300_EV]),
    kpoint_weights=np.full(epc.kpoints.shape[0], 1.0 / epc.kpoints.shape[0]),
    spin_degeneracy=2,
    dimension="2d",
    area=area,
)

scan_t = compute_serta_mobility_scan_si(
    eigenvalues=epc.eigenvalues_k,
    velocities=velocities,
    linewidth=linewidth,
    reciprocal_cell=reciprocal_cell,
    chemical_potentials=dirac_energy + MU_REL_EV,
    temperatures=TEMPERATURES_EV,
    kpoint_weights=np.full(epc.kpoints.shape[0], 1.0 / epc.kpoints.shape[0]),
    spin_degeneracy=2,
    dimension="2d",
    area=area,
)

carrier_cm2 = scan_300.carrier_density[:, 0] / 1e4
conductivity_s = np.trace(scan_300.conductivity[:, 0, :2, :2], axis1=1, axis2=2)
mobility_cm2_vs = np.trace(scan_300.mobility[:, 0, :2, :2], axis1=1, axis2=2) * 0.5 * 1e4

carrier_t_cm2 = scan_t.carrier_density / 1e4
mobility_t_cm2_vs = np.trace(scan_t.mobility[:, :, :2, :2], axis1=2, axis2=3) * 0.5 * 1e4
fixed_density_mobility = []
for it in range(TEMPERATURES_K.size):
    order = np.argsort(carrier_t_cm2[:, it])
    fixed_density_mobility.append(np.interp(TARGET_DENSITY_CM2, carrier_t_cm2[order, it], mobility_t_cm2_vs[order, it]))
fixed_density_mobility = np.asarray(fixed_density_mobility)

scan_300.save_npz(WORK / "fig4_mobility_scan_300k.npz")
scan_t.save_npz(WORK / "fig4_mobility_scan_temperature.npz")
print("carrier density cm^-2", carrier_cm2)
print("mobility cm^2/V/s", mobility_cm2_vs)


## Figure: carrier density, conductivity, mobility, temperature trend

This panel layout mirrors paper Fig. 4. The reduced mesh is not intended to reproduce the paper's absolute numbers; it verifies DeePTB's end-to-end API path and physical outputs.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(8.2, 6.0), dpi=140, constrained_layout=True)
ax = axes[0, 0]
ax.plot(MU_REL_EV, carrier_cm2, marker="o")
ax.set_xlabel("chemical potential above Dirac [eV]")
ax.set_ylabel("carrier density [cm^-2]")
ax.set_title("a. carrier density")
ax.grid(alpha=0.25)

ax = axes[0, 1]
ax.plot(MU_REL_EV, conductivity_s, marker="o", color="tab:orange")
ax.set_xlabel("chemical potential above Dirac [eV]")
ax.set_ylabel("sigma_xx + sigma_yy [S]")
ax.set_title("b. conductivity")
ax.grid(alpha=0.25)

ax = axes[1, 0]
ax.plot(carrier_cm2, mobility_cm2_vs, marker="o", color="tab:green")
ax.set_xlabel("carrier density [cm^-2]")
ax.set_ylabel("mobility [cm^2/V/s]")
ax.set_title("c. mobility versus density")
ax.grid(alpha=0.25)

ax = axes[1, 1]
ax.plot(TEMPERATURES_K, fixed_density_mobility, marker="o", color="tab:red")
ax.set_xlabel("temperature [K]")
ax.set_ylabel("mobility [cm^2/V/s]")
ax.set_title(f"d. mobility at n~{TARGET_DENSITY_CM2:.1e} cm^-2")
ax.grid(alpha=0.25)

fig.suptitle("Graphene SERTA transport, reduced-grid DeePTB reproduction")
fig.savefig(WORK / "figure_03_fig4_transport_mobility.png", dpi=200)
plt.show()


In [ ]:
summary = {
    "figure": "paper_fig4_style_transport_mobility",
    "chemical_potential_above_dirac_ev": MU_REL_EV,
    "carrier_density_cm2_300k": carrier_cm2,
    "conductivity_trace_s_300k": conductivity_s,
    "mobility_cm2_vs_300k": mobility_cm2_vs,
    "temperature_k": TEMPERATURES_K,
    "target_density_cm2": TARGET_DENSITY_CM2,
    "fixed_density_mobility_cm2_vs": fixed_density_mobility,
    "note": "Reduced-grid DeePTB-native reproduction of paper Fig. 4 panel structure; not a converged publication benchmark.",
}
(WORK / "fig4_transport_mobility_summary.json").write_text(json.dumps(summary, indent=2, default=json_default))
print(json.dumps(summary, indent=2, default=json_default))
